In [17]:
import networkx as nx
import pandas as pd
import random
import numpy as np
import json
from tqdm import tqdm
    
from pathlib import Path
import sys
    
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
base_path = PROJECT_ROOT / "experiment_new"

from cup_main_queue_data import color_update_propagation
from cup_classes import Iteration
from cup_functions import update_queue_cup, update_queue_cas, update_queue_cor
from hcgcr_gather_data import hcgcr_data

q_func_names = ["cup", "cas", "cor"]
update_functions = [
    update_queue_cup,
    update_queue_cas,
    update_queue_cor,
]


### Unpack random graphs

In [18]:
# Functions to unpack the data from csv files
def json_default(obj):
    """Translates numpy data types to native Python data types for JSON serialization"""
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, set):
        return list(obj)
    raise TypeError(f"Object of type {type(obj).__name__} is not JSON serializable")

def nxgraph(n, edges):
    G = nx.Graph()
    G.add_edges_from([tuple(e) for e in edges])
    G.add_nodes_from(range(int(n)))
    return G

def restore_dict_values(hash_dict):
    # hash_dict: {"hash": {"color": int, "orbit": list-or-set}, ...}
    output = {}
    for hash_str, val in hash_dict.items():
        
        hash_key = round(float(hash_str), 7)  # same precision as in json
        color = int(val["color"])
        orbit = val.get("orbit")
        
        if isinstance(orbit, list):
            orbit = set(orbit)

        output[hash_key] = {"color": color, "orbit": orbit}

    return output

def unpack(name):
    """Unpack the graph data from csv file with name 'name' and 
    return a dataframe with two columns: graph and iterations 
    (list of iteration objects).
    
    The files were generated in the file generate_random_graphs.ipynb,
    where the graph data was stored in a csv file with columns:
    - n: number of nodes in the graph
    - graph: list of edges in the graph
    - nr_iter: number of iterations in the color refinement process
    - color: list of colorings in each iteration (list of lists)
    - hash: list of hash values in each iteration (list of lists)
    - hash_dict: list of hash dictionaries in each iteration (list of dicts)
    """
    
    df = pd.read_csv(base_path / "test_graphs" / name)
    for col in ["graph", "color", "hash", "hash_dict"]:
        df[col] = df[col].apply(json.loads)
        
    graphs = []
    iterations = []
    for _, row in df.iterrows():
        graphs.append(nxgraph(row["n"], row["graph"]))
        iter = []
        for color_list, hash_list, hash_dict_from_json in zip(row["color"], row["hash"], row["hash_dict"]):
            color = np.array(color_list)
            hash = np.array(hash_list)
            hash_dict = restore_dict_values(hash_dict_from_json)
            iter.append(Iteration(color=color, hash=hash, hash_dict=hash_dict))
        iterations.append(iter)
    return pd.DataFrame({"graph": graphs, "iterations": iterations})

### Run CUP, CAS and COR algorithms on all graphs and save the results

In [19]:
def update_dict(graph_dict, old_iterations, colorings_up, q_lens, T_scr):
    """Create a dictionary with results of processing an updated graph."""
    
    T_up = sum(x != 0 for x in q_lens)
    T_old = len(old_iterations)
    n = graph_dict["graph_up"].number_of_nodes()
    t = min(T_up, T_old) # number of iterations that are in both processes
    k = len(graph_dict["changed_edges"])
    avg = (sum(q_lens[:t]) / t) if t > 0 else 0
    old_coloring = old_iterations[-1].color
    new_coloring = colorings_up[-1]
    nr_v_changed = sum(old_coloring != new_coloring)
    
    data = {
        "n": n,
        "m": graph_dict["graph_up"].number_of_edges(),
        "k": k, # number of changed edges
        "T_old": T_old,
        "T_up": T_up,
        "T_scr": T_scr,
        "T_delta": T_scr - T_up,
        "q_lens": json.dumps(q_lens),
        "q_len_average": avg,
        "W": sum(q_lens[:t]), # work done in the iterations that are in both processes
        "W_scr": n * T_scr, # work done in the static color refinement
        "v_changed": nr_v_changed,
        "changed_edges": json.dumps(list(graph_dict["changed_edges"])),
        "S": json.dumps(list(graph_dict["S"])),
        "colorings_up": [np.asarray(colorings_up[i]).tolist() for i in range(len(colorings_up))],

    }
    return data

def generate_updated_results(df, graphs_up, q_func, q_func_name, fname):
    """Generate the results of processing the updated graphs with update function 'q_func'
    and save them in a csv file with name 'fname'.

    Args:
        df (pd.DataFrame): a dataframe with original graphs and iterations (output of unpack function)
        graphs_up (dict): a list of dictionaries with updated graphs and related information (S, changed_edges)
        q_func (Callable): the update function to apply
        q_func_name (string): the name of the update function
        fname (string): the name of the output file
    """
    
    result = []
    for i, row in enumerate(tqdm(df.itertuples(index=False), desc=f"{q_func_name}")):
        
        graph_dict = graphs_up[i]
        G_up = graph_dict["graph_up"]
        S    = graph_dict["S"]
        
        old_iterations = row.iterations
        
        try:
            scr_iterations = hcgcr_data(G_up) # iterations of the static color refinement on the updated graph
            colorings_up, q_lens = color_update_propagation(G_up, S, old_iterations, q_func)
            d = update_dict(graph_dict, old_iterations, colorings_up, q_lens, len(scr_iterations))
            
        except Exception as e:
            # In case of an error, we log the error and save the data with empty colorings and q_lens
            print(f"Error in {q_func_name}, row {i}: {e},"
                f"{colorings_up}, {q_lens}")
            d = {        
                "n": G_up.number_of_nodes(),
                "m": G_up.number_of_edges(),
                "k": len(graph_dict["changed_edges"]), # number of changed edges
                "T_old": len(old_iterations),
                "T_up": json.dumps([]),
                "T_delta": json.dumps([]),
                "q_lens": json.dumps([]),
                "q_len_average": json.dumps([]),
                "W": json.dumps([]),
                "v_changed": json.dumps([]),
                "S": json.dumps(list(S)),
                "changed_edges": json.dumps(list(graph_dict["changed_edges"])),
                "colorings_up": json.dumps([])
            }
        
        result.append(d)
    pd.DataFrame(result).to_csv(base_path / "results" / f"result_{fname}_{q_func_name}.csv", index=False)


### 1. Remove one edge in a tree

In [20]:
df = unpack("random_trees.csv")

def remove_random_edge(G, seed=None):
    rng = random.Random(seed)
    pair = rng.sample(list(G.edges()), 1)[0]
    return {pair[0], pair[1]}

# draw deleted edges
S_list = [
    remove_random_edge(G, seed="0000")
    for G in df["graph"]
]
# prepare updated graphs - remove edges in copies of G
graphs_up = []
for (u, v), G in zip(S_list, df["graph"]):
    G_up = G.copy()
    G_up.remove_edge(u, v)
    graphs_up.append({
        "graph_up": G_up,
        "S": {u,v},
        "changed_edges": [(u,v)]
        })

for qfn, q_func in zip(q_func_names, update_functions):
    print("We go with the update function:", qfn)
    try:
        generate_updated_results(df, graphs_up, q_func, qfn, "trees")
    except Exception as e:
        print("Error:", e)

We go with the update function: cup


cup: 250it [00:03, 81.02it/s] 


We go with the update function: cas


cas: 250it [00:03, 78.95it/s] 


We go with the update function: cor


cor: 250it [00:03, 79.67it/s] 


### 2. Add and delete edges in G(n,p) random graphs

In [21]:
def generate_perturbed_graph(G, alpha, seed=None):
    """Add and delete some random edges in the graph G.
    Parameter alpha is the parameter of edges to add/delete.
    """
    rng = random.Random(seed)
    G_upd = G.copy()

    m = G.number_of_edges()
    k = int(alpha * m)
    if k == 0:
        # at least one edit, if 1 then we add, if 2 then we delete
        k = rng.randint(1,2)
        
    k_del = k // 2
    k_add = k - k_del

    # delete edges
    edges = list(G_upd.edges())
    del_edges = []
    if k_del > 0:
        del_edges = rng.sample(edges, min(k_del, len(edges)))
        G_upd.remove_edges_from(del_edges)

    # add edges
    non_edges = list(nx.non_edges(G_upd))

    # normalize edges for comparison
    del_edges_set = {tuple(sorted(e)) for e in del_edges}

    # filter out deleted edges
    non_edges = [e for e in non_edges if tuple(sorted(e)) not in del_edges_set]

    add_edges = []
    if k_add > 0 and non_edges:
        add_edges = rng.sample(non_edges, min(k_add, len(non_edges)))
        G_upd.add_edges_from(add_edges)

        return G_upd, (add_edges + del_edges)


In [22]:

for f in ["05", "1", "5"]:
    f_n = f"random_graphs_p_{f}.csv"
    df = unpack(f_n)
    print(f"Random graphs from a file: {f_n}")

    for a in [0.001, 0.005, 0.01]:
        graphs_up = []
        for G in df["graph"]:
            G_up, changed_edges = generate_perturbed_graph(G, a, "1111")
            graphs_up.append({
                "graph_up": G_up, 
                "S": {v for e in changed_edges for v in e}, 
                "changed_edges": changed_edges})

        for qfn, q_func in zip(q_func_names, update_functions):
            print(f"a = {a}, we go with the update function: {qfn}")
            try:
                generate_updated_results(df, graphs_up, q_func, qfn, f"rg_p_{f}_a_{int(1000*a)}")
            except Exception as e:
                print("Error:", e)

Random graphs from a file: random_graphs_p_05.csv
a = 0.001, we go with the update function: cup


cup: 250it [00:01, 138.78it/s]


a = 0.001, we go with the update function: cas


cas: 250it [00:02, 96.78it/s] 


a = 0.001, we go with the update function: cor


...

cup: 250it [00:04, 61.36it/s]


a = 0.01, we go with the update function: cas


cas: 250it [00:03, 70.94it/s]


a = 0.01, we go with the update function: cor


cor: 250it [00:03, 70.31it/s] 
